In [1]:
!python --version

Python 3.13.12


In [2]:
# notebook for ml model dataset creation and training
from autogluon.tabular import TabularDataset, TabularPredictor
import json
import csv

In [3]:
JSON_DIR = "../../openmvcam-firmware/mission-29/scans.json"
CSV_DIR = "csv-dat/testds-{plant}.csv"
WL = [410, 435, 460, 485, 510, 535, 560, 585, 610, 645, 680, 705, 730, 760, 810, 860, 900, 940]
with open(JSON_DIR, 'r') as f:
    scans_dict = json.load(f)

In [4]:
ds_dict = {"all": []}
for tag_id, entry in scans_dict["entries"].items():
    plant_rf = []
    for sample_cts, ref_cts in zip(entry["scanwl"], entry["calwl"]):
        if ref_cts != 0:
            plant_rf.append(sample_cts / ref_cts)
        else:
            plant_rf.append(0)

    ds_dict["all"].append([tag_id, entry["plant"] if entry["plant"] is not None else "null", entry["prox"], entry["amb"]] + plant_rf)
    if entry["plant"] not in ds_dict:
        ds_dict[entry["plant"]] = []
    ds_dict[entry["plant"]].append([tag_id, entry["prox"], entry["amb"]] + plant_rf)

for plant_name, dataset in ds_dict.items():
    with open(CSV_DIR.format(plant = plant_name if plant_name is not None else "null"), 'w') as f:
        csv_writer = csv.writer(f)
        if plant_name == "all":
            csv_writer.writerow(["tag_id", "plant_name", "prox", "amb"] + WL)
        else:
            csv_writer.writerow(["tag_id", "prox", "amb"] + WL)
        csv_writer.writerows(dataset)